# 📖 Module 04: Embeddings Deep Dive

## GenAI L2 Exam Preparation

**Topics Covered:**
- What are embeddings and why they matter
- Embedding models comparison (HuggingFace, Google, OpenAI)
- Dimensions and distance metrics
- Cosine similarity explained
- Hands-on: Generate and compare embeddings

**Source Material:** Class 29, Class 31 (Embeddings in RAG context)

---

## 1. What are Embeddings?

Embeddings are **numerical vector representations** of text that capture **semantic meaning**.

```
"I love dogs"  →  [0.12, -0.34, 0.78, ..., 0.56]   (384 dimensions)
"I adore puppies" → [0.11, -0.33, 0.77, ..., 0.55]  (very similar!)
"The stock market crashed" → [0.89, 0.12, -0.45, ..., -0.23] (very different!)
```

### Key Properties
- **Semantic similarity**: Similar meanings → similar vectors → close in vector space
- **Fixed dimensions**: Each model produces vectors of a fixed size (384, 768, 1536, etc.)
- **Dense**: Unlike sparse representations (BOW, TF-IDF), every dimension has a non-zero value

### Role in RAG
```
Text Chunks → Embedding Model → Vectors → Vector DB → Similarity Search ← Query Embedding
```

### 🎯 Exam Tip
> Embeddings enable **semantic search** — finding documents by **meaning**, not just keyword matching.  
> "dog food" can match "pet nutrition" because their embeddings are close in vector space.

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()
print("✅ Environment loaded")

## 2. Embedding Models Comparison

| Model | Provider | Dimensions | Cost | Speed | Quality |
|-------|----------|-----------|------|-------|--------|
| `all-MiniLM-L6-v2` | HuggingFace | 384 | Free ⭐ | Fast | Good |
| `all-mpnet-base-v2` | HuggingFace | 768 | Free | Medium | Better |
| `models/embedding-001` | Google | 768 | Free tier | Fast (API) | Good |
| `text-embedding-3-small` | OpenAI | 1536 | Paid | Fast (API) | Very Good |
| `text-embedding-3-large` | OpenAI | 3072 | Paid | Medium (API) | Best |

### Recommendation for Exam / Development
1. **Default choice**: `all-MiniLM-L6-v2` (free, runs locally, good quality)
2. **Need better quality**: `all-mpnet-base-v2` (free, runs locally, better quality)
3. **Production**: `text-embedding-3-small` (OpenAI, excellent quality)

### 🎯 Exam Tip
> **Critical rule**: The embedding model used for **indexing** must be the **same** as the one used for **querying**.  
> If you embed documents with model A (384d) and query with model B (768d), it will **not work**.

## 3. Hands-On: HuggingFace Embeddings (Free, Local)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize embedding model (runs locally, no API key needed)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Embed a single text
text = "RAG combines retrieval with generation"
vector = embeddings.embed_query(text)

print(f"📝 Text: '{text}'")
print(f"📐 Vector dimension: {len(vector)}")
print(f"🔢 First 10 values: {vector[:10]}")
print(f"📊 Min: {min(vector):.4f}, Max: {max(vector):.4f}")

In [ ]:
# Embed multiple texts at once
texts = [
    "RAG combines retrieval with generation",
    "Retrieval-Augmented Generation enhances LLM responses",
    "The weather today is sunny and warm",
    "Vector databases store embeddings for fast search",
    "I love eating pizza for dinner"
]

# embed_documents is for batch embedding
vectors = embeddings.embed_documents(texts)

print(f"📊 Embedded {len(vectors)} texts")
print(f"📐 Each vector has {len(vectors[0])} dimensions")

### 🎯 Exam Tip
> Two important methods:  
> - `embed_query(text)` → for embedding a **single query** string  
> - `embed_documents(texts)` → for embedding a **batch of documents** (list of strings)  
> Some models optimize differently for queries vs documents.

## 4. Distance Metrics & Cosine Similarity

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Compare similarity between our texts
print("📊 COSINE SIMILARITY MATRIX")
print("=" * 60)
print()

# Short labels for display
labels = ["RAG+retrieval", "RAG+LLM", "Weather", "VectorDB", "Pizza"]

for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        sim = cosine_similarity(vectors[i], vectors[j])
        emoji = "🟢" if sim > 0.5 else "🟡" if sim > 0.3 else "🔴"
        print(f"{emoji} {labels[i]:>15} vs {labels[j]:<15}: {sim:.4f}")

print()
print("🟢 High similarity (>0.5) | 🟡 Medium (0.3-0.5) | 🔴 Low (<0.3)")

### Distance Metrics Explained

| Metric | Range | Interpretation | Used By |
|--------|-------|---------------|---------|
| **Cosine Similarity** | -1 to 1 | 1 = identical, 0 = orthogonal, -1 = opposite | Most vector DBs ⭐ |
| **Euclidean Distance** | 0 to ∞ | 0 = identical, higher = more different | FAISS |
| **Dot Product** | -∞ to ∞ | Higher = more similar | Some specialized cases |

### 🎯 Exam Tip
> **Cosine similarity** is the **most commonly used** metric in RAG.  
> It measures the **angle** between vectors, not the magnitude.  
> This means it works well even if vectors have different lengths (magnitudes).

## 5. ⭐ Embedding Dimension Mismatch (Common Exam Scenario)

One of the **most common debugging scenarios** in RAG:

```
ERROR: Dimension mismatch — expected 384, got 768
```

### What causes it:
- Indexed documents with Model A (384d), querying with Model B (768d)
- Changed embedding model without re-indexing the vector store
- Different model versions producing different dimensions

### How to fix:
1. **Always use the same model** for indexing and querying
2. When changing models, **re-index all documents**
3. Check dimension with `len(embeddings.embed_query('test'))`

### 🎯 Exam Tip
> If an exam question describes retrieval returning errors or no results after a model change,  
> the answer is likely **embedding dimension mismatch** — need to re-index.

In [ ]:
# Demonstrate: Check embedding dimensions
print("📐 EMBEDDING MODEL DIMENSIONS")
print("=" * 40)

# HuggingFace all-MiniLM-L6-v2
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dim = len(hf_embeddings.embed_query("test"))
print(f"all-MiniLM-L6-v2:    {dim} dimensions")

# You can also check with other models if available
print(f"\n💡 Always verify dimensions match between indexing and querying!")

## 6. Dense vs Sparse Embeddings

| Feature | Dense Embeddings | Sparse Embeddings |
|---------|-----------------|-------------------|
| **Method** | Neural network models | TF-IDF, BM25 |
| **Vector** | All values non-zero | Most values are zero |
| **Captures** | Semantic meaning | Keyword frequency |
| **Example** | "dog" ≈ "puppy" | "dog" ≠ "puppy" |
| **Strength** | Understanding meaning | Exact keyword matching |
| **Weakness** | Can miss exact keywords | Misses synonyms/meaning |
| **Used in** | Vector search (FAISS, Chroma) | Traditional search (BM25) |

### Hybrid Search
Combines both for best results:
```
Hybrid Score = α × Dense Score + (1-α) × Sparse Score
```

### 🎯 Exam Tip
> Dense embeddings catch **semantic similarity** ("happy" ≈ "joyful")  
> Sparse representations catch **keyword matches** ("Python" = "Python")  
> **Hybrid search** = best of both worlds

## 🧠 Self-Assessment Quiz

---

**Q1.** What is the dimension of vectors produced by `all-MiniLM-L6-v2`?

<details>
<summary>Click for Answer</summary>

**384 dimensions**. This is a compact model that balances quality and speed.
</details>

---

**Q2.** You indexed documents using `all-MiniLM-L6-v2` (384d) but now want to use `text-embedding-3-small` (1536d) for queries. Will this work?

<details>
<summary>Click for Answer</summary>

**No!** This will cause a dimension mismatch error. You must **re-index all documents** with the new embedding model. The indexing and querying models must produce vectors of the **same dimension**.
</details>

---

**Q3.** What is the difference between `embed_query()` and `embed_documents()`?

<details>
<summary>Click for Answer</summary>

- `embed_query(text)` — embeds a **single string** (the user's query)  
- `embed_documents(texts)` — embeds a **list of strings** (batch of documents)  
Some models optimize differently for queries vs documents (asymmetric embedding).
</details>

---

**Q4.** Cosine similarity between two vectors is 0.95. What does this mean?

<details>
<summary>Click for Answer</summary>

The two texts are **very semantically similar** (0.95 out of 1.0). Their vectors point in nearly the same direction in the embedding space. This would be a strong match in retrieval.
</details>

---

**Q5.** Why might hybrid search (dense + sparse) outperform pure dense search?

<details>
<summary>Click for Answer</summary>

Dense embeddings capture **semantic meaning** but can miss exact keyword matches. Sparse methods (BM25) excel at **exact keyword matching** but miss synonyms. Hybrid search combines both, so it catches both "Python programming" (exact match) and "coding in the Python language" (semantic match).
</details>

---

## ✅ Module 4 Complete!

**Key Takeaways:**
1. Embeddings convert text to vectors that capture semantic meaning
2. Same model must be used for indexing AND querying
3. Default: `all-MiniLM-L6-v2` (384d, free, local)
4. Cosine similarity is the standard distance metric
5. Dense captures meaning, sparse captures keywords, hybrid = best
6. Dimension mismatch is the #1 embedding debugging issue

**Next:** [Module 05 — Vector Databases](./05_Vector_Databases.ipynb)